# Travel-Mode Constraint Function

## Overview

This notebook defines and validates a reusable age-based travel-mode constraint function.

For profiles aged 0–4 or 5–11, `Car or van driver` and `Motorcycle` are removed from each travel-mode distribution. The remaining probabilities are then renormalised to sum to one.

Profiles from the demographic profile pool are used to test the function. The test profiles are validation cases only and do not represent population sampling.

## 1. Import Libraries

In [1]:
import json

from copy import deepcopy
from pathlib import Path

import pandas as pd

## 2. Set Up File Paths

In [2]:
project_root = Path.cwd().parents[1]

population_file = (
    project_root
    / "output"
    / "demographic_profiles"
    / "representative_population.parquet"
)

behavioural_evidence_file = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence.json"
)

## 3. Load Input Data

In [3]:
population = pd.read_parquet(population_file)

with open(
    behavioural_evidence_file,
    "r",
    encoding="utf-8",
) as file:
    behavioural_evidence = json.load(file)

print("Population loaded:", population.shape)
print("Trip purposes loaded:", len(behavioural_evidence))

Population loaded: (2869120, 14)
Trip purposes loaded: 14


## 4. Define Travel-Mode Constraint Function

### Travel-Mode Constraint

The function accepts an age category and one travel-mode probability distribution.

If the age is 0–4 or 5–11, `Car or van driver` is removed and the remaining probabilities are renormalised. Other age groups retain the original distribution.

In [4]:
def apply_travel_mode_constraint(
    age,
    travel_mode_probabilities,
):
    constrained = deepcopy(travel_mode_probabilities)

    if age not in {"0-4", "5-11"}:
        return constrained

    constrained.pop("Car or van driver", None)
    constrained.pop("Motorcycle", None)

    total = sum(constrained.values())

    if total <= 0:
        raise ValueError(
            "No feasible travel-mode probability remains."
        )

    return {
        mode: probability / total
        for mode, probability in constrained.items()
    }

## 5. Select Test Profiles

In [5]:
test_ages = [
    "0-4",
    "5-11",
    "23-34",
]

profile_columns = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

test_profiles = []

for age in test_ages:

    candidates = population[
        population["Age"] == age
    ]

    if candidates.empty:
        raise ValueError(
            f"No feasible profile found for age={age}."
        )

    test_profiles.append(
        candidates.iloc[0][profile_columns].to_dict()
    )

test_profiles_df = pd.DataFrame(test_profiles)

display(test_profiles_df)

,Age,Gender,Household Type,Economic Activity Status,Industry,Ethnic Group,Distance travelled to work,Car Ownership
0,0-4,Female,Couple household: With dependent children,Does not apply,Does not apply,"Asian, Asian British or Asian Welsh: Bangladeshi",Does not apply,1 car or van in household
1,5-11,Female,Couple household: With dependent children,Does not apply,Does not apply,Mixed or Multiple ethnic groups: Other Mixed o...,Does not apply,1 car or van in household
2,23-34,Female,Couple household: All children non-dependent,Economically active and a full-time student: I...,J Information and communication,"Asian, Asian British or Asian Welsh: Other Asian",10km to less than 30km,1 car or van in household


## 6. Test and Validate Function

In [6]:
validation_summary = []

for profile in test_profiles:

    age = profile["Age"]
    mode_sets = []
    probability_sums_valid = True
    original_modes = None

    for purpose, evidence in behavioural_evidence.items():

        record = evidence["travel_mode_profile"][0]

        original = {
            mode: probability
            for mode, probability in record.items()
            if mode != "Purpose"
        }

        constrained = apply_travel_mode_constraint(
            age=age,
            travel_mode_probabilities=original,
        )

        if original_modes is None:
            original_modes = set(original)

        mode_sets.append(set(constrained))

        if abs(sum(constrained.values()) - 1) >= 1e-8:
            probability_sums_valid = False

    remaining_modes = mode_sets[0]

    assert all(
        modes == remaining_modes
        for modes in mode_sets
    )

    removed_modes = (
        original_modes - remaining_modes
    )

    if age in {"0-4", "5-11"}:
        assert removed_modes == {
            "Car or van driver",
            "Motorcycle"
        }
    else:
        assert not removed_modes

    assert probability_sums_valid

    validation_summary.append(
        {
            "Age": age,
            "Travel Mode Count": len(
                remaining_modes
            ),
            "Removed Travel Mode": (
                ", ".join(sorted(removed_modes))
                if removed_modes
                else "None"
            ),
            "Remaining Travel Modes": (
                ", ".join(sorted(remaining_modes))
            ),
            "Probability Validation": "PASS",
        }
    )

validation_summary = pd.DataFrame(
    validation_summary
)

display(validation_summary)

,Age,Travel Mode Count,Removed Travel Mode,Remaining Travel Modes,Probability Validation
0,0-4,9,"Car or van driver, Motorcycle","Car or van passenger, Non-local bus, Other Pri...",PASS
1,5-11,9,"Car or van driver, Motorcycle","Car or van passenger, Non-local bus, Other Pri...",PASS
2,23-34,11,None,"Car or van driver, Car or van passenger, Motor...",PASS


## Summary

The reusable travel-mode constraint function was successfully validated using feasible profiles aged 0–4, 5–11, and 23–34.

For ages 0–4 and 5–11, `Car or van driver` and `Motorcycle`  were removed from every trip-purpose-conditioned travel-mode distribution and the remaining probabilities were renormalised. The adult control profile retained its original travel-mode distributions.